# Python 기초 통합 프로젝트
## Part 2. 판매 점검 코드를 함수화하고 결과 저장하기

### Part 1과의 연결

Part 1에서 작성한 판매금액 구간 분류와 배송 확인 거래 검색 코드를 함수로 바꾸어 반복 사용이 가능한 구조로 개선합니다.

### 실무 시나리오

영업관리팀은 매일 새로운 판매 데이터에 같은 점검 기준을 적용합니다. 데이터 담당자는 반복 코드를 함수로 정리하고, 배송 확인 대상 거래를 CSV 파일로 저장하여 담당자에게 전달해야 합니다.

### 과제 목표

- 반복 코드를 재사용 가능한 함수로 작성할 수 있습니다.
- 매개변수, 반환값, 기본값을 사용할 수 있습니다.
- `TypeError`, `ValueError`, `PermissionError` 등 오류 유형을 구분해 처리할 수 있습니다.
- 분석 결과를 CSV 파일로 저장하고 다시 확인할 수 있습니다.
- 심화 문제에서 고액 반품 결과를 JSON으로 저장할 수 있습니다.

### 사용 환경

- 결과물: `.ipynb` 파일 1개
- 필수 생성 파일: `delivery_check_sales.csv`
- 심화 생성 파일: `high_value_return_sales.json`
- 사용 언어: Python 3.X
- 사용 라이브러리: pandas

## 제공 코드. 데이터 불러오기

In [2]:
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

print("전체 거래 수:", len(df))

전체 거래 수: 500


# 문제 1. 판매금액 구간 분류 함수 만들기

## 함수

```python
classify_sales_by_amount(sale_ids, sale_amounts)
```

## 요구사항

1. 두 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 두 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 데이터가 비어 있으면 `ValueError`를 발생시킵니다.
4. 고가·중가·일반 거래의 `SaleID`를 딕셔너리로 반환합니다.
5. 함수를 호출하고 결과를 출력합니다.

In [3]:
# 문제 1 코드를 작성하세요.
def classify_sales_by_amount(sale_ids, sale_amounts):
    high = {}
    middle = {}
    low = {}
    if type(sale_ids)!=list or type(sale_amounts) != list:
        raise TypeError
    elif (len(sale_ids) != len(sale_amounts)) or (len(sale_amounts) ==0 or len(sale_ids) ==0) :
        raise ValueError
    else:
        for i in range (len(sale_amounts)):
            if sale_amounts[i]>=70000000:
                high[i] = sale_ids[i]
            elif sale_amounts[i]>=40000000:
                middle[i] = sale_ids[i]
            else:
                low[i] = sale_ids[i]

    return high,middle,low

high,middle,low = classify_sales_by_amount(sale_ids,sale_amounts)

print(f" 고가:{high} \n 중가:{middle} \n 일반:{low}")

 고가:{1: 'S2025010064', 3: 'S2025010264', 15: 'S2025010093', 16: 'S2025010135', 25: 'S2025010483', 29: 'S2025010165', 31: 'S2025020115', 49: 'S2025020161', 55: 'S2025020030', 66: 'S2025020413', 75: 'S2025020437', 77: 'S2025020031', 79: 'S2025030414', 83: 'S2025030139', 90: 'S2025030005', 99: 'S2025030049', 102: 'S2025040389', 109: 'S2025040494', 117: 'S2025040336', 118: 'S2025040344', 128: 'S2025040284', 129: 'S2025040369', 131: 'S2025040429', 135: 'S2025040262', 136: 'S2025040288', 140: 'S2025040206', 143: 'S2025040178', 158: 'S2025050451', 167: 'S2025050444', 172: 'S2025050127', 177: 'S2025050303', 179: 'S2025050183', 186: 'S2025050260', 191: 'S2025050273', 198: 'S2025050401', 201: 'S2025060015', 211: 'S2025060430', 212: 'S2025060153', 220: 'S2025060072', 232: 'S2025060189', 233: 'S2025060217', 234: 'S2025060331', 243: 'S2025060490', 264: 'S2025070458', 269: 'S2025070220', 274: 'S2025070073', 276: 'S2025070474', 279: 'S2025070083', 291: 'S2025080218', 293: 'S2025080209', 300: 'S202508

# 문제 2. 배송 확인 거래 검색 함수 만들기

## 함수

```python
find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
)
```

## 요구사항

1. 세 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 세 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 배송일 20일 이상이면서 출고완료가 아닌 거래의 `SaleID`를 반환합니다.
4. 정상 입력과 잘못된 입력을 각각 테스트합니다.
5. `TypeError`와 `ValueError`를 별도의 `except`에서 처리합니다.

In [4]:
# 문제 2 코드를 작성하세요.
def find_delivery_check_sales(sale_ids,delivery_days,stock_statuses):
    sale = []
    try:
        if not isinstance(sale_ids, list) or not isinstance(delivery_days, list) or not isinstance(stock_statuses,list):
            raise TypeError
        elif len(sale_ids) != len(delivery_days) or len(delivery_days) !=len(stock_statuses) or len(sale_ids) !=len(stock_statuses):
            raise ValueError
        else:
            for i in range(len(sale_ids)):
                if delivery_days[i]>=20 and stock_statuses[i] != "출고완료":
                    sale.append(sale_ids[i])
    except TypeError:
        print("세 입력값이 리스트가 아닙니다.")
    except ValueError:
        print("세 리스트의 길이가 다릅니다.")
    finally:
        return sale


# 정상 입력
result = find_delivery_check_sales(sale_ids,delivery_days,stock_statuses)
print("배송일 20일 이상 걸렸지만 출고가 완료되지 않은 거래 id:\n",result)

# 비 정상 입력
result2 = find_delivery_check_sales(sale_ids,delivery_days,1)



배송일 20일 이상 걸렸지만 출고가 완료되지 않은 거래 id:
 ['S2025010207', 'S2025010051', 'S2025010165', 'S2025020062', 'S2025020216', 'S2025020065', 'S2025020357', 'S2025030117', 'S2025030095', 'S2025030466', 'S2025040178', 'S2025040452', 'S2025050446', 'S2025050352', 'S2025050100', 'S2025050444', 'S2025050111', 'S2025050493', 'S2025050241', 'S2025050401', 'S2025060310', 'S2025060269', 'S2025060072', 'S2025060295', 'S2025060047', 'S2025060032', 'S2025060374', 'S2025070155', 'S2025070033', 'S2025070457', 'S2025070035', 'S2025070328', 'S2025080070', 'S2025080145', 'S2025080194', 'S2025090350', 'S2025090067', 'S2025100460', 'S2025100316', 'S2025100106', 'S2025110197', 'S2025110222', 'S2025110496', 'S2025110238', 'S2025120500', 'S2025120476', 'S2025120002', 'S2025120340']
세 입력값이 리스트가 아닙니다.


# 문제 3. 배송 점검 결과를 CSV로 저장하기

## 요구사항

1. 문제 2에서 반환된 `SaleID`로 원본 DataFrame의 거래를 선택합니다.
2. `delivery_check_sales.csv`로 저장합니다.
3. 파일 인덱스는 저장하지 않습니다.
4. 저장 파일을 다시 불러와 거래 수를 비교합니다.
5. `PermissionError`와 그 외 `OSError`를 구분하여 처리합니다.

In [5]:
# 문제 3 코드를 작성하세요.
df_result = df[df['SaleID'].isin(result)]

try:
    #저장
    df_result.to_csv("delivery_check_sales.csv", index=False)
    # 다시 불러오기
    re_df = pd.read_csv("delivery_check_sales.csv")
except PermissionError:
    print("파일에 접근할 수 없습니다.")
except OSError:
    print("파일을 저장하거나 불러오는 중 오류가 발생했습니다.")

print("원본 거래 수:", len(df_result))
print("불러온 거래 수:", len(re_df))

원본 거래 수: 48
불러온 거래 수: 48


# 심화 문제. 고액 반품 검색 및 JSON 저장

이 문제는 **5점 수준을 위한 선택 문제**입니다.

## 요구사항

1. `find_high_value_returns()` 함수를 작성합니다.
2. `min_amount=70_000_000` 기본값을 사용합니다.
3. 입력 자료형이 잘못되면 `TypeError`, 기준값이 잘못되면 `ValueError`를 발생시킵니다.
4. 고액 반품 거래를 JSON 파일로 저장합니다.
5. 저장한 JSON을 다시 읽어 거래 수를 확인합니다.
6. `PermissionError`, `OSError`, `ValueError`를 구분해 처리합니다.

In [6]:
# 심화 문제 코드를 작성하세요.
import json

def find_high_value_returns(sale_ids, sale_amounts, returned_flags, min_amount=70_000_000):
    ValueError
    if not isinstance(sale_ids, list) or not isinstance(sale_amounts, list) or not isinstance(returned_flags, list):
        raise TypeError('list 자료형이어야 합니다.')

    if not isinstance(min_amount, (int,float)):
        raise TypeError("기준값은 int or float 자료형이어야 합니다.")

    if min_amount < 0:
        raise ValueError('min_amount는 0이상이어야 합니다.')

    if len(sale_ids) != len(sale_amounts) or  len(sale_ids) != len(returned_flags):
        raise ValueError('입력값 리스트의 길이가 같아야 합니다.')

    high_returns = []
    for i in range(len(sale_ids)):
        if sale_amounts[i] >= min_amount and returned_flags[i] == 'Y':
            high_returns.append({
                'SaleID': sale_ids[i],
                'FinalSaleAmount': sale_amounts[i],
                'IsReturned': returned_flags[i]
            })

    try: 
        with open('high_return.json', "w", encoding='utf-8') as file:
            json.dump(high_returns, file, ensure_ascii=False, indent=4 )
        print('파일 저장완료')

    except PermissionError:
        return ('파일에 접근할 수 없습니다.')
    except OSError:
        return ('파일을 저장하는 중 오류가 발생했습니다.')

    try:
        with open('high_return.json', 'r', encoding='utf-8') as file: 
            load = json.load(file)
        print(f'읽어온 저장 건수: {len(load)}')
    except PermissionError:
        return ('파일에 접근할 수 없습니다.')
    except OSError:
        return ('파일을 불러올 수 없습니다.')


find_high_value_returns(sale_ids, sale_amounts, returned_flags)




파일 저장완료
읽어온 저장 건수: 3


# 제출 결과물

| 결과물 | 구분 |
|---|---|
| 판매금액 구간 분류 함수 | 필수 |
| 배송 확인 거래 함수 | 필수 |
| `TypeError`, `ValueError` 구분 처리 | 필수 |
| 배송 확인 CSV 저장 및 재확인 | 필수 |
| `PermissionError`, `OSError` 구분 처리 | 필수 |
| 고액 반품 함수와 JSON 저장 | 심화 |